In [1]:
import os
print(os.getcwd())

C:\Users\DELL\Logistic-regression-practice-project


In [2]:
import pandas as pd

In [3]:
df = pd.read_csv(r"C:\Users\DELL\Downloads\loan_default_data.csv")

In [5]:
df.shape

(10000, 14)

In [6]:
df.columns

Index(['customer_id', 'age', 'annual_income', 'credit_score', 'loan_amount',
       'loan_tenure_months', 'interest_rate', 'dti', 'utilization',
       'days_past_due', 'inquiries_6m', 'existing_loans', 'employment_type',
       'default'],
      dtype='object')

In [8]:
df["default"].value_counts(dropna=False)

default
0    8574
1    1426
Name: count, dtype: int64

In [9]:
df.isna().sum()

customer_id             0
age                     0
annual_income         200
credit_score          150
loan_amount             0
loan_tenure_months      0
interest_rate           0
dti                   100
utilization             0
days_past_due           0
inquiries_6m            0
existing_loans          0
employment_type         0
default                 0
dtype: int64

In [10]:
df.dtypes

customer_id            object
age                     int64
annual_income         float64
credit_score          float64
loan_amount             int64
loan_tenure_months      int64
interest_rate         float64
dti                   float64
utilization           float64
days_past_due           int64
inquiries_6m            int64
existing_loans          int64
employment_type        object
default                 int64
dtype: object

In [11]:
features = [
    "age",
    "annual_income",
    "credit_score",
    "loan_amount",
    "loan_tenure_months",
    "interest_rate",
    "dti",
    "utilization",
    "inquiries_6m",
    "existing_loans",
    "employment_type"
]

X = df[features].copy()
y = df["default"].copy()

In [12]:
print(X.shape)

(10000, 11)


In [13]:
print(y.shape)

(10000,)


In [14]:
print("Repeated customer IDs:", df["customer_id"].duplicated().sum())

Repeated customer IDs: 0


In [19]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split (
    X,
    y,
    test_size = 0.20,
    random_state = 42,
    stratify = y
)

In [21]:
X_train.shape

(8000, 11)

In [22]:
X_test.shape

(2000, 11)

In [23]:
y_train.shape

(8000,)

In [24]:
y_test.shape

(2000,)

In [28]:
y_train.value_counts(normalize = True).mul(100)

default
0    85.7375
1    14.2625
Name: proportion, dtype: float64

In [29]:
y_test.value_counts(normalize = True).mul(100)

default
0    85.75
1    14.25
Name: proportion, dtype: float64

In [31]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   customer_id         10000 non-null  object 
 1   age                 10000 non-null  int64  
 2   annual_income       9800 non-null   float64
 3   credit_score        9850 non-null   float64
 4   loan_amount         10000 non-null  int64  
 5   loan_tenure_months  10000 non-null  int64  
 6   interest_rate       10000 non-null  float64
 7   dti                 9900 non-null   float64
 8   utilization         10000 non-null  float64
 9   days_past_due       10000 non-null  int64  
 10  inquiries_6m        10000 non-null  int64  
 11  existing_loans      10000 non-null  int64  
 12  employment_type     10000 non-null  object 
 13  default             10000 non-null  int64  
dtypes: float64(5), int64(7), object(2)
memory usage: 1.1+ MB


In [38]:
numeric_features = X_train.select_dtypes(
    include="number"
).columns.tolist()
categorical_features = ["employment_type"]

In [35]:
df["employment_type"]

0            Salaried
1            Salaried
2            Salaried
3       Self-employed
4            Salaried
            ...      
9995         Salaried
9996         Salaried
9997    Self-employed
9998         Salaried
9999         Salaried
Name: employment_type, Length: 10000, dtype: object

In [36]:
print("Numerical:", numeric_features)
print("Categorical:", categorical_features)

Numerical: ['age', 'annual_income', 'credit_score', 'loan_amount', 'loan_tenure_months', 'interest_rate', 'dti', 'utilization', 'inquiries_6m', 'existing_loans']
Categorical: employment_type


In [39]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [40]:
from sklearn.preprocessing import OneHotEncoder

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [41]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

In [46]:
from sklearn.linear_model import LogisticRegression

lr_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        C = 1.0,
        solver="lbfgs",
        max_iter=1000
    )
    )
])

In [47]:
lr_pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [51]:
y_pred = lr_pipeline.predict(X_test)
y_prob = lr_pipeline.predict_proba(X_test)[:, 1]

In [52]:
results = pd.DataFrame({
    "actual_default": y_test,
    "predicted_default": y_pred,
    "predicted_PD": y_prob
})

results.head(10)

,actual_default,predicted_default,predicted_PD
5567,0,0,0.147964
3255,0,0,0.071559
7235,0,0,0.091680
7344,0,0,0.312334
2161,0,0,0.099627
4261,0,0,0.123731
9230,0,0,0.085656
919,0,0,0.221980
6506,0,0,0.029963
8697,0,0,0.450004


In [53]:
print(len(X_test))          # 2000 rows
print(X_test.index[:5])     # Original labels

X_test.head(5)

2000
Index([5567, 3255, 7235, 7344, 2161], dtype='int64')


,age,annual_income,credit_score,loan_amount,loan_tenure_months,interest_rate,dti,utilization,inquiries_6m,existing_loans,employment_type
5567,36,580710.0,683.0,459462,24,17.59,0.274,0.362,2,4,Contract
3255,29,926650.0,695.0,92481,36,15.32,0.134,0.524,0,3,Self-employed
7235,28,759847.0,699.0,100868,60,16.88,0.099,0.643,2,1,Self-employed
7344,54,464783.0,653.0,99704,60,21.56,0.409,0.844,1,3,Salaried
2161,31,697118.0,712.0,658369,36,13.08,0.426,0.371,2,0,Self-employed


In [54]:
results = results.reset_index(drop=True)
results.head(10)

,actual_default,predicted_default,predicted_PD
0,0,0,0.147964
1,0,0,0.071559
2,0,0,0.091680
3,0,0,0.312334
4,0,0,0.099627
5,0,0,0.123731
6,0,0,0.085656
7,0,0,0.221980
8,0,0,0.029963
9,0,0,0.450004


In [55]:
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Confusion matrix:
[[1692   23]
 [ 258   27]]

Classification report:
              precision    recall  f1-score   support

           0       0.87      0.99      0.92      1715
           1       0.54      0.09      0.16       285

    accuracy                           0.86      2000
   macro avg       0.70      0.54      0.54      2000
weighted avg       0.82      0.86      0.81      2000

ROC-AUC: 0.7501754385964913


In [56]:
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.25,
    random_state=42,
    stratify=y_train
)

print("Fit:", X_fit.shape)
print("Validation:", X_val.shape)

Fit: (6000, 11)
Validation: (2000, 11)


In [57]:
from sklearn.base import clone

validation_model = clone(lr_pipeline)
validation_model.fit(X_fit, y_fit)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [58]:
val_prob = validation_model.predict_proba(X_val)[:, 1]

print(val_prob[:10])

[0.16557356 0.04568504 0.20908051 0.04016659 0.03104463 0.09250182
 0.2690341  0.07658531 0.13745956 0.06245368]


In [59]:
from sklearn.metrics import precision_score, recall_score, f1_score

threshold_results = []

for threshold in [0.10, 0.20, 0.30, 0.40, 0.50]:
    val_pred = (val_prob >= threshold).astype(int)

    threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(y_val, val_pred, zero_division=0),
        "recall": recall_score(y_val, val_pred, zero_division=0),
        "f1": f1_score(y_val, val_pred, zero_division=0),
        "predicted_defaults": val_pred.sum()
    })

threshold_table = pd.DataFrame(threshold_results)
threshold_table

,threshold,precision,recall,f1,predicted_defaults
0,0.1,0.226937,0.863158,0.359386,1084
1,0.2,0.353556,0.592982,0.442988,478
2,0.3,0.463768,0.336842,0.390244,207
3,0.4,0.552941,0.164912,0.254054,85
4,0.5,0.689655,0.070175,0.127389,29


In [61]:
val_pred = (val_prob >= threshold).astype(int)

In [62]:
chosen_threshold = 0.20

test_prob_final = validation_model.predict_proba(X_test)[:, 1]
test_pred_final = (
    test_prob_final >= chosen_threshold
).astype(int)

print("Confusion matrix:")
print(confusion_matrix(y_test, test_pred_final))

print("\nClassification report:")
print(classification_report(
    y_test, test_pred_final, zero_division=0
))

print("ROC-AUC:", roc_auc_score(y_test, test_prob_final))

Confusion matrix:
[[1403  312]
 [ 123  162]]

Classification report:
              precision    recall  f1-score   support

           0       0.92      0.82      0.87      1715
           1       0.34      0.57      0.43       285

    accuracy                           0.78      2000
   macro avg       0.63      0.69      0.65      2000
weighted avg       0.84      0.78      0.80      2000

ROC-AUC: 0.7519103882154365


In [63]:
calibration_df = pd.DataFrame({
    "predicted_PD": test_prob_final,
    "actual_default": y_test.to_numpy()
})

calibration_df["PD_group"] = pd.qcut(
    calibration_df["predicted_PD"],
    q=5,
    duplicates="drop"
)

calibration_table = (
    calibration_df.groupby("PD_group", observed=True)
    .agg(
        customers=("actual_default", "size"),
        average_PD=("predicted_PD", "mean"),
        actual_default_rate=("actual_default", "mean")
    )
)

calibration_table

,customers,average_PD,actual_default_rate
PD_group,,,
"(0.00368, 0.0555]",400,0.036307,0.0325
"(0.0555, 0.0907]",400,0.072163,0.0775
"(0.0907, 0.14]",400,0.114912,0.1025
"(0.14, 0.226]",400,0.175220,0.1300
"(0.226, 0.844]",400,0.351735,0.3700


In [64]:
feature_names = (
    validation_model.named_steps["preprocessor"]
    .get_feature_names_out()
)

coefficients = (
    validation_model.named_steps["model"]
    .coef_[0]
)

coefficient_table = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients
}).sort_values("coefficient")

coefficient_table

,feature,coefficient
11,cat__employment_type_Salaried,-0.719650
2,num__credit_score,-0.632976
12,cat__employment_type_Self-employed,-0.550736
10,cat__employment_type_Contract,-0.247142
0,num__age,-0.037826
1,num__annual_income,-0.014108
4,num__loan_tenure_months,0.023134
5,num__interest_rate,0.032802
9,num__existing_loans,0.080637
13,cat__employment_type_Unemployed,0.086646
